# Sliding Puzzle AI Solver based on DeepCubeA (4x4 - 7x7) v0.14 beta

In [ ]:
#@title Solver
import os, sys, subprocess, pickle, numpy as np
from IPython.display import display

# --- versions ---
import pip
print(f"Python {sys.version}")
print(f"pip {pip.__version__}")
print(f"NumPy {np.__version__}")

try:
    import torch
    print(f"PyTorch {torch.__version__}")
except:
    print("PyTorch not installed")

# --- setup ---
REPO_URL = "https://github.com/dphdmn/DeepCubeA.git"
if not os.path.exists("DeepCubeA"):
    !git clone --quiet {REPO_URL} 2>/dev/null
    %cd DeepCubeA

!pip install -q -r requirements.txt 2>/dev/null
!bash compile.sh 2>/dev/null

from solve import solve_single
import ipywidgets as widgets


def on_solve_click(b):
    with out:
        out.clear_output()
        scrambles = [s.strip() for s in scramble_input.value.strip().splitlines() if s.strip()]
        if not scrambles:
            print("No scrambles entered.")
            return
        solutions = []
        errors = 0
        for i, sc in enumerate(scrambles):
            print(f"\n=== Scramble {i+1}/{len(scrambles)} ===")
            try:
                sol = solve_single(sc, batch_size_w.value, language=language_w.value)
                if sol:
                    solutions.append(sol)
            except Exception as e:
                print(f"Error: {e}")
                errors += 1
        if errors:
            print(f"\n{errors}/{len(scrambles)} failed.")
        else:
            print(f"\nAll {len(scrambles)} done.")
        if solutions:
            sol_file = "solutions.txt"
            with open(sol_file, 'w') as f:
                for s in solutions:
                    f.write(s + '\n')
            print(f"\nSolutions saved to {sol_file}. Opening viewer...")
            from google.colab import files
            files.view(sol_file)


title = widgets.HTML("<h3>Solver</h3>")

scramble_input = widgets.Textarea(
    value='12 6 4 9 18/3 5 17 11 7/13 22 14 24 10/23 2 21 1 20/19 0 16 8 15',
    placeholder='Enter scrambles, one per line...',
    layout=widgets.Layout(width='100%', height='150px')
)

language_w = widgets.Dropdown(options=['python', 'cpp'], value='python', description='Solver:', layout=widgets.Layout(width='200px'))
batch_size_w = widgets.IntText(value=4000, description='Batch size:')

solve_btn = widgets.Button(description='Solve', button_style='primary')
solve_btn.on_click(on_solve_click)

out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px', width='100%'))

display(widgets.VBox([title, scramble_input, widgets.HBox([language_w, batch_size_w]), solve_btn, out]))